# 3 - Book Recommendation System - Recommendation Systems

<img src='https://assets.penguinrandomhouse.com/wp-content/uploads/2016/01/11104627/21Books-PRH_site_1200x628.jpg'>



Bu çalışmada kullanıcıların kitap puanlarına göre benzer kitapları öneren bir recommendation system geliştireceğiz.

## Akış

1. Veriyi yükleme
2. Veriyi okuma ve inceleme
3. Veri temizleme
4. Feature engineering
5. User-book matrix oluşturma
6. Kitap benzerliklerini hesaplama
7. Kitap öneri fonksiyonu oluşturma
8. Sonuç

In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

## 1. Veriyi Yükleme

In [2]:
# Bu bölümde zip dosyasını Google Drive içinden açıp çalışma alanına çıkaracağım.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/Book Recommendation Dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content')

os.listdir('/content')[:20]

Mounted at /content/drive


['.config',
 'Ratings.csv',
 'classicRec.png',
 'DeepRec.png',
 'drive',
 'recsys_taxonomy2.png',
 'Books.csv',
 'Users.csv',
 'sample_data']

## 2. Veriyi Okuma ve İnceleme

In [5]:
# Bu bölümde kitap, kullanıcı ve puan verilerini okuyup veri setlerinin yapısını inceleyeceğim.

In [12]:
books = pd.read_csv('/content/Books.csv', sep=',', encoding='latin1', on_bad_lines='skip')
ratings = pd.read_csv('/content/Ratings.csv', sep=',', encoding='latin1', on_bad_lines='skip')
users = pd.read_csv('/content/Users.csv', sep=',', encoding='latin1', on_bad_lines='skip')

books.head()

/tmp/ipykernel_1526/440999993.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('/content/Books.csv', sep=',', encoding='latin1', on_bad_lines='skip')


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [13]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [14]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [15]:
books.shape, ratings.shape, users.shape

((271360, 8), (1149780, 3), (278858, 3))

## 3. Veri Temizleme

In [16]:
# Bu bölümde sütun adlarını sadeleştirip recommendation sistemi için gerekli alanları seçeceğim.

In [17]:
books = books[['ISBN', 'Book-Title', 'Book-Author']]
ratings = ratings[['User-ID', 'ISBN', 'Book-Rating']]

books.columns = ['ISBN', 'Title', 'Author']
ratings.columns = ['UserID', 'ISBN', 'Rating']

books.head()

,ISBN,Title,Author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


In [18]:
ratings = ratings[ratings['Rating'] > 0]
ratings.isnull().sum()

,0
UserID,0
ISBN,0
Rating,0


## 4. Feature Engineering

In [19]:
# Bu bölümde kitap ve puan verilerini birleştirip yeterli etkileşime sahip kullanıcı ve kitapları filtreleyeceğim.

In [20]:
df = ratings.merge(books, on='ISBN')
df.head()

,UserID,ISBN,Rating,Title,Author
0,276726,0155061224,5,Rites of Passage,Judith Rae
1,276729,052165615X,3,Help!: Level 1,Philip Prowse
2,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather
3,276744,038550120X,7,A Painted House,JOHN GRISHAM
4,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells


In [21]:
book_rating_counts = df['Title'].value_counts()
active_books = book_rating_counts[book_rating_counts >= 50].index

df = df[df['Title'].isin(active_books)]

df.shape

(65477, 5)

In [22]:
user_rating_counts = df['UserID'].value_counts()
active_users = user_rating_counts[user_rating_counts >= 50].index

df = df[df['UserID'].isin(active_users)]

df.shape

(3273, 5)

## 5. User-book Matrix Oluşturma

In [23]:
# Bu bölümde kullanıcı-kitap puan matrisini oluşturacağım.

In [24]:
book_pivot = df.pivot_table(columns='UserID', index='Title', values='Rating').fillna(0)
book_pivot.head()

UserID,6575,7346,11676,16795,21014,23902,31315,31826,35859,37712,...,189835,204864,210485,219683,225087,236283,240567,258534,261829,271448
Title,,,,,,,,,,,,,,,,,,,,,
1984,0.0,8.0,10.000000,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,7.0,0.0,0.0,8.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,9.000000,9.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,8.0,0.0
2nd Chance,0.0,0.0,7.500000,0.0,0.0,0.0,10.0,0.0,7.5,0.0,...,0.0,9.0,0.0,0.0,10.0,0.0,0.0,0.0,7.5,0.0
4 Blondes,0.0,0.0,0.000000,0.0,0.0,7.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,8.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
book_pivot.shape

(641, 41)

## 6. Kitap Benzerliklerini Hesaplama

In [26]:
# Bu bölümde kitaplar arasındaki benzerliği cosine similarity ile hesaplayacağım.

In [27]:
similarity_scores = cosine_similarity(book_pivot)
similarity_scores.shape

(641, 641)

In [28]:
similarity_df = pd.DataFrame(similarity_scores, index=book_pivot.index, columns=book_pivot.index)
similarity_df.iloc[:5, :5]

Title,1984,1st to Die: A Novel,2nd Chance,4 Blondes,84 Charing Cross Road
Title,,,,,
1984,1.000000,0.393629,0.108810,0.0,0.413096
1st to Die: A Novel,0.393629,1.000000,0.360925,0.0,0.354375
2nd Chance,0.108810,0.360925,1.000000,0.0,0.263401
4 Blondes,0.000000,0.000000,0.000000,1.0,0.000000
84 Charing Cross Road,0.413096,0.354375,0.263401,0.0,1.000000


## 7. Kitap Öneri Fonksiyonu Oluşturma

In [29]:
# Bu bölümde seçilen bir kitap için en benzer kitapları döndüren recommendation fonksiyonunu oluşturacağım.

In [30]:
def recommend_books(book_name, top_n=5):
    similar_books = similarity_df[book_name].sort_values(ascending=False)[1:top_n+1]
    recommendations = pd.DataFrame({
        'Title': similar_books.index,
        'Similarity': similar_books.values
    })
    return recommendations

In [31]:
# Bu bölümde veri setindeki örnek bir kitap için öneri sonuçlarını göstereceğim.

In [32]:
example_book = book_pivot.index[0]
example_book

'1984'

In [33]:
recommend_books(example_book, top_n=5)

,Title,Similarity
0,The Virgin Suicides,0.689717
1,The Devil Wears Prada : A Novel,0.670868
2,The Hot Zone,0.670350
3,Call of the Wild,0.670350
4,Congo,0.667748


## 8. Sonuç

Bu projede kullanıcıların kitap puanlarına göre benzer kitapları önermek için item-based recommendation yöntemi kullanıldı. Elde edilen sonuçlara göre seçilen kitap için benzer 5 kitap başarıyla önerildi.